In [55]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split,cross_validate,StratifiedKFold
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE,ADASYN
from imblearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,make_scorer,cohen_kappa_score,matthews_corrcoef


In [56]:
#Read CSV file
data=pd.read_csv("bank-additional-full.csv",sep=";")

In [57]:
#Drop Leakage feature
data=data.drop('duration',axis=1)

In [58]:
#Handling "Unknown" values
AllColumns=data.select_dtypes(include='object').columns
for col in AllColumns:
  data[col]=data[col].replace('unknown',data[col].mode()[0])

In [59]:
#Label Encoding
data['y']=data['y'].map({'yes':1,'no':0})

In [60]:
# One Hot Encoding
data=pd.get_dummies(data,drop_first=True)

In [61]:
#Feature and Target Spliting
X=data.drop('y',axis=1)
y=data['y']

In [62]:
#Train Test Split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

In [63]:
#Scaling
scaler=StandardScaler()
scaler.fit_transform(X_train)
scaler.transform(X_test)

array([[-0.77033007,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [-0.28972159, -0.56702251,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 3.17065947, -0.20368791,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       ...,
       [-0.67420837, -0.56702251,  0.19658384, ..., -0.4964409 ,
        -2.50346033, -0.18627755],
       [ 0.38313029,  1.61298507,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755],
       [ 0.19088689,  0.88631588,  0.19658384, ..., -0.4964409 ,
         0.39944711, -0.18627755]])

In [64]:
#sampling techniques
smote=SMOTE(random_state=42)
adasyn=ADASYN(random_state=42)

In [77]:
#Evaluation Matrices
kappa=make_scorer(cohen_kappa_score)
mcc= make_scorer(matthews_corrcoef)
Scoring={
    'accuracy':'accuracy',
    'precision':'precision',
    'recall':'recall',
    'f1':'f1',
    'roc_auc':'roc_auc',
    'kappa':kappa,
    'mcc':mcc
}

In [66]:
#models
models={
    "Decision_Tree":DecisionTreeClassifier(random_state=42),
    "Random_Forest":RandomForestClassifier(random_state=42),
    "Logistic_Regression":LogisticRegression(random_state=42),
    "KNN":KNeighborsClassifier(),
    "MLP":MLPClassifier(random_state=42),
    "XGBoost":XGBClassifier(random_state=42)
}

In [67]:
#Cross validation setup
skf=StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [78]:
def Model_evaluation(X,y,Sampler,label):
  print("========{label}========")
  for name,model in models.items():
    pipeline=Pipeline([('sampler',Sampler),('model',model)])
    results=cross_validate(pipeline,X,y,cv=skf,scoring=Scoring)
    print(f"\nModel: {name}")
    print(f"Accuracy:  {np.mean(results['test_accuracy']):.4f}")
    print(f"Precision: {np.mean(results['test_precision']):.4f}")
    print(f"Recall:    {np.mean(results['test_recall']):.4f}")
    print(f"F1 Score:  {np.mean(results['test_f1']):.4f}")
    print(f"ROC-AUC:   {np.mean(results['test_roc_auc']):.4f}")
    print(f"Kappa:    {np.mean(results['test_kappa']):.4f}")
    print(f"MCC:    {np.mean(results['test_mcc']):.4f}")

In [79]:
Model_evaluation(X,y,smote,"SMOTE")

========{label}========

Model: Decision_Tree
Accuracy:  0.8296
Precision: 0.2954
Recall:    0.3700
F1 Score:  0.3285
ROC-AUC:   0.6307
Kappa:    0.2323
MCC:    0.2343

Model: Random_Forest
Accuracy:  0.8797
Precision: 0.4608
Recall:    0.3981
F1 Score:  0.4271
ROC-AUC:   0.7687
Kappa:    0.3603
MCC:    0.3615


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c


Model: Logistic_Regression
Accuracy:  0.8049
Precision: 0.3044
Recall:    0.5668
F1 Score:  0.3955
ROC-AUC:   0.7505
Kappa:    0.2918
MCC:    0.3122

Model: KNN
Accuracy:  0.8267
Precision: 0.3270
Recall:    0.5080
F1 Score:  0.3978
ROC-AUC:   0.7334
Kappa:    0.3021
MCC:    0.3122

Model: MLP
Accuracy:  0.7048
Precision: 0.3405
Recall:    0.4726
F1 Score:  0.2693
ROC-AUC:   0.7280
Kappa:    0.1663
MCC:    0.2119

Model: XGBoost
Accuracy:  0.8858
Precision: 0.4917
Recall:    0.4037
F1 Score:  0.4432
ROC-AUC:   0.7754
Kappa:    0.3803
MCC:    0.3826
